In [ ]:
# Install required libraries
!pip install -q transformers datasets librosa evaluate jiwer

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
import os
import pandas as pd
import torch
import librosa
from transformers import pipeline
from tqdm import tqdm
import re



In [ ]:
import os
import pandas as pd
import torch
import librosa
import re
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from tqdm import tqdm

# ==========================================
# 1. SETUP PATHS
# ==========================================
INPUT_PATH = "/kaggle/input/competitions/multilingual-speech-recognition"
TEST_CSV = os.path.join(INPUT_PATH, "test.csv")
DATA_DIR = os.path.join(INPUT_PATH, "competition_data", "test")

test_df = pd.read_csv(TEST_CSV)

# ==========================================
# 2. LOAD MODEL & PROCESSOR MANUALLY
# ==========================================
print("--- Loading Whisper Model & Processor ---")
model_id = "openai/whisper-base"

# Load processor (handles audio -> features) and model (handles features -> text)
processor = WhisperProcessor.from_pretrained(model_id)
model = WhisperForConditionalGeneration.from_pretrained(model_id)

# Force to CPU for maximum stability given the previous CUDA errors
device = torch.device("cpu")
model.to(device)

# ==========================================
# 3. TRANSCRIPTION LOOP
# ==========================================
print("\n--- Starting Transcription (Direct Method) ---")

predictions = []

for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
    audio_path = os.path.join(DATA_DIR, row['audio'])
    
    try:
        # STEP A: Load audio with librosa (the standard way)
        speech, _ = librosa.load(audio_path, sr=16000)
        
        # STEP B: Convert audio to input features for the model
        input_features = processor(speech, sampling_rate=16000, return_tensors="pt").input_features 
        
        # STEP C: Generate token IDs (the actual "inference")
        # forced_decoder_ids=None tells it to predict the language automatically
        predicted_ids = model.generate(input_features.to(device))
        
        # STEP D: Decode IDs back into text
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        
        # STEP E: Clean the text
        clean_text = transcription.lower().strip()
        clean_text = re.sub(r'[^\w\s]', '', clean_text)
        
        predictions.append(clean_text)
        
        if i % 20 == 0:
            print(f" -> {row['audio']}: {clean_text[:50]}")
            
    except Exception as e:
        print(f"Error at index {i} ({row['audio']}): {e}")
        predictions.append("")

# ==========================================
# 4. EXPORT
# ==========================================
submission = pd.DataFrame({
    "audio": test_df["audio"],
    "text": predictions
})

submission.to_csv("submission.csv", index=False)
print("\n--- DONE! submission.csv created ---")